# Homework: Agentic RAG – Modul 1

Hausaufgabe: https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/01-agentic-rag/homework.md

Abgabe: https://courses.datatalks.club/llm-zoomcamp-2026/homework/hw1

## Setup – Umgebung vorbereiten

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

## Preparation – Daten laden (Commit 8c1834d)

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []
for file in files:
    doc = file.parse()
    documents.append(doc)

print(f"Geladene Dokumente: {len(documents)}")

## Q1. Wie viele Lesson-Seiten gibt es?

In [ ]:
print(f"Q1 Antwort – Anzahl Lesson-Seiten: {len(documents)}")

## Q2. Indexieren und Suchen

In [ ]:
import minsearch

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(documents)

results = index.search(
    query="How does the agentic loop keep calling the model until it stops?",
    num_results=5
)

print(f"Q2 Antwort – filename des ersten Ergebnisses: {results[0]['filename']}")

## Q3. RAG – Input Tokens zählen

In [ ]:
def build_context(results):
    context = ""
    for doc in results:
        context += f"Filename: {doc['filename']}\nContent: {doc['content']}\n\n"
    return context.strip()

def build_prompt(query, context):
    return f"""You are a helpful course assistant. Answer the question using only the context below.

Context:
{context}

Question: {query}""".strip()

def llm_with_usage(prompt):
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content, response.usage

def rag(query, search_index):
    results = search_index.search(query=query, num_results=5)
    context = build_context(results)
    prompt = build_prompt(query, context)
    answer, usage = llm_with_usage(prompt)
    return answer, usage

query = "How does the agentic loop keep calling the model until it stops?"
answer_q3, usage_q3 = rag(query, index)

print(f"Antwort: {answer_q3[:200]}...")
print(f"\nQ3 Antwort – Input Tokens: {usage_q3.prompt_tokens}")

## Q4. Chunking – Wie viele Chunks?

In [ ]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print(f"Q4 Antwort – Anzahl Chunks: {len(chunks)}")

## Q5. RAG mit Chunking – Vergleich Input Tokens

In [ ]:
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)

answer_q5, usage_q5 = rag(query, chunk_index)

ratio = usage_q3.prompt_tokens / usage_q5.prompt_tokens
print(f"Q3 Input Tokens (ohne Chunking): {usage_q3.prompt_tokens}")
print(f"Q5 Input Tokens (mit Chunking):  {usage_q5.prompt_tokens}")
print(f"Q5 Antwort – Faktor: {ratio:.1f}x weniger Tokens")

## Q6. Agent – Wie oft ruft er search auf?

In [ ]:
import toyaikit

search_call_count = 0

def search(query: str) -> list:
    """Search the course lessons for information about the given query."""
    global search_call_count
    search_call_count += 1
    print(f"  -> search() #{search_call_count}: '{query}'")
    results = chunk_index.search(query=query, num_results=3)
    return [{"filename": r["filename"], "content": r["content"][:500]} for r in results]

agent = toyaikit.Agent(
    client=openai_client,
    model="gpt-4o-mini",
    tools=[search],
    system_prompt=(
        "You're a course teaching assistant. Answer the student's question using the "
        "search tool. Make multiple searches with different keywords before answering."
    )
)

result = agent.run("How does the agentic loop work, and how is it different from plain RAG?")

print(f"\nAntwort: {result[:300]}...")
print(f"\nQ6 Antwort – Anzahl search()-Aufrufe: {search_call_count}")

## Zusammenfassung aller Antworten

In [ ]:
print("=" * 45)
print("HAUSAUFGABE – ALLE ANTWORTEN")
print("=" * 45)
print(f"Q1 – Anzahl Lesson-Seiten:       {len(documents)}")
print(f"Q2 – Filename erstes Ergebnis:   {results[0]['filename']}")
print(f"Q3 – Input Tokens (ohne Chunk):  {usage_q3.prompt_tokens}")
print(f"Q4 – Anzahl Chunks:              {len(chunks)}")
print(f"Q5 – Input Tokens (mit Chunk):   {usage_q5.prompt_tokens}")
print(f"Q6 – search()-Aufrufe:           {search_call_count}")
print("=" * 45)
print("Abgabe: https://courses.datatalks.club/llm-zoomcamp-2026/homework/hw1")